# Sistema RAG: Agente de Asesinos Seriales

Sistema RAG profesional para consultar información sobre asesinos seriales desde documentos PDF.

**Características:**
- Procesamiento de múltiples PDFs
- Búsqueda semántica con embeddings multilengua gratuitos
- Memoria conversacional con LangGraph
- Interfaz gráfica moderna y académica
- Integración con Groq LLM


## 1. Configuración e Importaciones

Importar todas las dependencias necesarias y configurar las variables principales del sistema.


In [38]:
# Instalación de dependencias (ejecutar si hay errores de importación)
%pip install chromadb langchain-groq langchain-text-splitters sentence-transformers pypdf

# Para instalar todas las dependencias (alternativa, descomentar si es necesario):
# %pip install -q langchain langchain-community langgraph chromadb langchain-groq pypdf sentence-transformers ipywidgets python-dotenv tiktoken torch

import os
import sys
from pathlib import Path
from typing import TypedDict, List, Annotated
import operator

# LangChain y LangGraph
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END

# ChromaDB
import chromadb

# Widgets para interfaz gráfica
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# Utilidades
from datetime import datetime
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Cargar variables de entorno desde .env
load_dotenv()

print("✓ Módulos importados correctamente")

Note: you may need to restart the kernel to use updated packages.
✓ Módulos importados correctamente



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
# =============================================================================
# CONFIGURACIÓN DEL SISTEMA
# =============================================================================

# Cargar variables de entorno explícitamente desde el directorio base
BASE_DIR = Path.cwd()
env_path = BASE_DIR / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path, override=True)
    print(f"✓ Archivo .env encontrado en: {env_path}")
else:
    # Intentar cargar desde la ubicación por defecto
    load_dotenv(override=True)
    if not (BASE_DIR / ".env").exists():
        print(f"⚠ Archivo .env no encontrado en: {env_path}")
        print(f"  Por favor, crea un archivo .env con: GROQ_API_KEY=tu-api-key-aqui")

# API Key de Groq (cargar desde .env o variable de entorno del sistema)
# Intentar múltiples nombres de variable por compatibilidad
GROQ_API_KEY = (
    os.getenv("GROQ_API_KEY") or 
    os.getenv("groq_api_key") or 
    os.getenv("GROQAPIKEY") or
    os.getenv("API_KEY") or
    "tu-api-key-de-groq-aqui"
)

# Configuración de directorios (BASE_DIR ya está definido arriba)
PDFS_DIR = BASE_DIR / "pdfs"
CHROMA_DB_DIR = BASE_DIR / "chroma_db"

# Crear directorios si no existen
PDFS_DIR.mkdir(exist_ok=True)
CHROMA_DB_DIR.mkdir(exist_ok=True)

# Configuración de embeddings
EMBEDDING_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"  # Modelo multilengua gratuito
EMBEDDING_DEVICE = "cpu"  # Cambiar a "cuda" si tienes GPU

# Configuración de chunking
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Configuración de Groq
GROQ_MODEL = "llama-3.3-70b-versatile"  # Modelo actual de Groq (reemplazo de llama-3.1-70b-versatile)
# Otros modelos disponibles: "llama-3.1-8b-instant", "mixtral-8x7b-32768", "gemma-7b-it"
GROQ_TEMPERATURE = 0.3  # Temperatura más baja para respuestas más precisas

# Configuración de RAG
NUM_RETRIEVAL_DOCS = 5  # Número de documentos a recuperar para el contexto

print("✓ Configuración completada")
print(f"  - Directorio de PDFs: {PDFS_DIR}")
print(f"  - Directorio de BD vectorial: {CHROMA_DB_DIR}")
print(f"  - Modelo de embeddings: {EMBEDDING_MODEL}")
# Verificar si la API key fue cargada (sin mostrar la clave completa)
if GROQ_API_KEY and GROQ_API_KEY != "tu-api-key-de-groq-aqui":
    print(f"  - GROQ_API_KEY: {'✓ Configurada (' + GROQ_API_KEY[:10] + '...)' if len(GROQ_API_KEY) > 10 else '✓ Configurada'}")
else:
    print(f"  - GROQ_API_KEY: ⚠ No configurada (verifica tu archivo .env)")

✓ Archivo .env encontrado en: c:\Users\martin\Desktop\RAG_Agente_Serial_Killers\.env
✓ Configuración completada
  - Directorio de PDFs: c:\Users\martin\Desktop\RAG_Agente_Serial_Killers\pdfs
  - Directorio de BD vectorial: c:\Users\martin\Desktop\RAG_Agente_Serial_Killers\chroma_db
  - Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2
  - GROQ_API_KEY: ✓ Configurada (gsk_Vq9LOu...)


## 2. Módulo de Procesamiento de PDFs

Funciones para cargar, procesar y dividir documentos PDF en chunks con metadatos.


In [40]:
def load_pdfs_from_directory(pdf_directory: Path) -> List[dict]:
    """
    Carga todos los PDFs de un directorio y los procesa.
    
    Args:
        pdf_directory: Path al directorio que contiene los PDFs
        
    Returns:
        Lista de documentos con texto y metadatos
    """
    documents = []
    pdf_files = list(pdf_directory.glob("*.pdf"))
    
    if not pdf_files:
        print(f"⚠ No se encontraron archivos PDF en {pdf_directory}")
        return documents
    
    print(f"📚 Procesando {len(pdf_files)} archivo(s) PDF...")
    
    for pdf_path in pdf_files:
        try:
            print(f"  - Procesando: {pdf_path.name}")
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
            
            for page in pages:
                # Agregar metadatos adicionales
                page.metadata['source_file'] = pdf_path.name
                page.metadata['file_path'] = str(pdf_path)
                documents.append(page)
                
            print(f"    ✓ {len(pages)} página(s) procesada(s)")
            
        except Exception as e:
            print(f"    ✗ Error procesando {pdf_path.name}: {str(e)}")
            continue
    
    print(f"✓ Total de documentos cargados: {len(documents)}")
    return documents


def chunk_documents(documents: List[dict], chunk_size: int = CHUNK_SIZE, 
                   chunk_overlap: int = CHUNK_OVERLAP) -> List[dict]:
    """
    Divide los documentos en chunks más pequeños con overlap.
    
    Args:
        documents: Lista de documentos a dividir
        chunk_size: Tamaño de cada chunk en caracteres
        chunk_overlap: Overlap entre chunks en caracteres
        
    Returns:
        Lista de chunks con metadatos preservados
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks = text_splitter.split_documents(documents)
    
    # Agregar índice de chunk a los metadatos
    for idx, chunk in enumerate(chunks):
        chunk.metadata['chunk_index'] = idx
        chunk.metadata['total_chunks'] = len(chunks)
    
    print(f"✓ Documentos divididos en {len(chunks)} chunks")
    return chunks

print("✓ Funciones de procesamiento de PDFs definidas")

✓ Funciones de procesamiento de PDFs definidas


In [41]:
# Inicializar embeddings multilengua gratuitos
print(f"📥 Cargando modelo de embeddings: {EMBEDDING_MODEL}")
print("   (Esto puede tardar unos minutos la primera vez...)")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': EMBEDDING_DEVICE},
    encode_kwargs={'normalize_embeddings': True}
)

print("✓ Modelo de embeddings cargado correctamente")

📥 Cargando modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2
   (Esto puede tardar unos minutos la primera vez...)
✓ Modelo de embeddings cargado correctamente


In [42]:
def get_processed_pdfs_from_vectorstore(vectorstore: Chroma) -> set:
    """
    Obtiene la lista de archivos PDF que ya están procesados en la base de datos.
    
    Args:
        vectorstore: Base de datos vectorial
        
    Returns:
        Set con los nombres de los archivos PDF ya procesados
    """
    processed_files = set()
    try:
        # Obtener todos los documentos de la colección
        all_docs = vectorstore._collection.get()
        if all_docs and 'metadatas' in all_docs:
            for metadata in all_docs['metadatas']:
                if metadata and 'source_file' in metadata:
                    processed_files.add(metadata['source_file'])
    except Exception as e:
        print(f"⚠ Error obteniendo PDFs procesados: {e}")
    return processed_files


def detect_new_pdfs(pdf_directory: Path, processed_files: set) -> List[Path]:
    """
    Detecta PDFs nuevos que no están en la base de datos.
    
    Args:
        pdf_directory: Directorio donde están los PDFs
        processed_files: Set con nombres de archivos ya procesados
        
    Returns:
        Lista de paths a PDFs nuevos
    """
    all_pdfs = list(pdf_directory.glob("*.pdf"))
    new_pdfs = [pdf for pdf in all_pdfs if pdf.name not in processed_files]
    return new_pdfs


def create_or_load_vectorstore(chunks: List[dict] = None, 
                                persist_directory: Path = CHROMA_DB_DIR,
                                collection_name: str = "serial_killers_rag",
                                add_to_existing: bool = False) -> Chroma:
    """
    Crea una nueva base de datos vectorial o carga una existente.
    Puede agregar nuevos documentos a una BD existente.
    
    Args:
        chunks: Lista de chunks para crear la BD (None si solo se quiere cargar)
        persist_directory: Directorio donde persistir la BD
        collection_name: Nombre de la colección en ChromaDB
        add_to_existing: Si True y existe BD, agrega chunks a la existente
        
    Returns:
        Objeto Chroma (vectorstore)
    """
    # Si no se proporcionan chunks, intentar cargar una BD existente
    if chunks is None:
        print(f"📂 Intentando cargar base de datos vectorial existente...")
        try:
            vectorstore = Chroma(
                persist_directory=str(persist_directory),
                embedding_function=embeddings,
                collection_name=collection_name
            )
            doc_count = vectorstore._collection.count()
            print(f"✓ Base de datos vectorial cargada ({doc_count} documentos)")
            return vectorstore
        except Exception as e:
            print(f"⚠ No se encontró base de datos existente o error al cargar: {e}")
            raise ValueError("No hay base de datos existente y no se proporcionaron chunks para crear una nueva")
    
    if len(chunks) == 0:
        raise ValueError("La lista de chunks está vacía")
    
    # Si add_to_existing es True, intentar cargar BD existente y agregar chunks
    if add_to_existing:
        try:
            vectorstore = Chroma(
                persist_directory=str(persist_directory),
                embedding_function=embeddings,
                collection_name=collection_name
            )
            print(f"📝 Agregando {len(chunks)} nuevos chunks a la base de datos existente...")
            print(f"   - Esto puede tardar varios minutos...")
            vectorstore.add_documents(chunks)
            doc_count = vectorstore._collection.count()
            print(f"✓ Chunks agregados exitosamente. Total de documentos: {doc_count}")
            return vectorstore
        except Exception as e:
            print(f"⚠ No se pudo agregar a BD existente: {e}")
            print(f"   Creando nueva base de datos...")
    
    print(f"🔄 Creando nueva base de datos vectorial...")
    print(f"   - Chunks a procesar: {len(chunks)}")
    print(f"   - Esto puede tardar varios minutos...")
    
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(persist_directory),
        collection_name=collection_name
    )
    
    print(f"✓ Base de datos vectorial creada exitosamente")
    return vectorstore


def retrieve_relevant_documents(vectorstore: Chroma, query: str, 
                                k: int = NUM_RETRIEVAL_DOCS) -> List[dict]:
    """
    Recupera documentos relevantes para una consulta.
    
    Args:
        vectorstore: Base de datos vectorial
        query: Consulta del usuario
        k: Número de documentos a recuperar
        
    Returns:
        Lista de documentos relevantes
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    # Usar invoke() para versiones recientes de LangChain
    try:
        docs = retriever.invoke(query)
    except AttributeError:
        # Fallback para versiones anteriores
        docs = retriever.get_relevant_documents(query)
    return docs

print("✓ Funciones de vectorstore definidas")

✓ Funciones de vectorstore definidas


## 4. Procesamiento de PDFs (Detección Automática)

Esta celda detecta automáticamente PDFs nuevos y los procesa. 
- Si es la primera vez, procesa todos los PDFs y crea la base de datos.
- Si ya existe una BD, detecta PDFs nuevos y los agrega automáticamente.
- Ejecuta esta celda cada vez que agregues nuevos PDFs al directorio `pdfs/`.


In [43]:
# Cargar y procesar PDFs con detección automática de nuevos
print("=" * 60)
print("PROCESAMIENTO DE PDFs")
print("=" * 60)

# Intentar cargar vectorstore existente primero
vectorstore = None
processed_files = set()

try:
    vectorstore = create_or_load_vectorstore(chunks=None)
    processed_files = get_processed_pdfs_from_vectorstore(vectorstore)
    print(f"   - PDFs ya procesados: {len(processed_files)}")
    if processed_files:
        print(f"   - Archivos: {', '.join(list(processed_files)[:5])}{'...' if len(processed_files) > 5 else ''}")
except:
    print("   - No existe base de datos previa, se creará una nueva")

# Detectar PDFs nuevos
pdf_files_in_dir = list(PDFS_DIR.glob("*.pdf"))
new_pdfs = detect_new_pdfs(PDFS_DIR, processed_files)

print(f"\n📊 Resumen:")
print(f"   - PDFs en directorio: {len(pdf_files_in_dir)}")
print(f"   - PDFs ya procesados: {len(processed_files)}")
print(f"   - PDFs nuevos detectados: {len(new_pdfs)}")

if new_pdfs:
    print(f"\n🆕 Procesando {len(new_pdfs)} PDF(s) nuevo(s):")
    for pdf in new_pdfs:
        print(f"   - {pdf.name}")
    
    # Procesar solo los PDFs nuevos
    new_documents = []
    for pdf_path in new_pdfs:
        try:
            print(f"\n📄 Procesando: {pdf_path.name}")
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
            
            for page in pages:
                page.metadata['source_file'] = pdf_path.name
                page.metadata['file_path'] = str(pdf_path)
                new_documents.append(page)
            
            print(f"   ✓ {len(pages)} página(s) procesada(s)")
        except Exception as e:
            print(f"   ✗ Error procesando {pdf_path.name}: {str(e)}")
            continue
    
    if new_documents:
        # Dividir en chunks
        new_chunks = chunk_documents(new_documents, CHUNK_SIZE, CHUNK_OVERLAP)
        
        # Agregar a la BD existente o crear nueva
        if vectorstore is not None:
            vectorstore = create_or_load_vectorstore(new_chunks, add_to_existing=True)
            print("\n✓ PDFs nuevos agregados a la base de datos existente")
        else:
            vectorstore = create_or_load_vectorstore(new_chunks)
            print("\n✓ Sistema RAG inicializado correctamente")
    else:
        print("\n⚠ No se pudieron procesar los PDFs nuevos")
        if vectorstore is None:
            vectorstore = None
elif not pdf_files_in_dir:
    print("\n⚠ No se encontraron archivos PDF en el directorio.")
    print(f"   Por favor, coloca archivos PDF en: {PDFS_DIR}")
    if vectorstore is None:
        vectorstore = None
else:
    print("\n✓ Todos los PDFs ya están procesados. No hay archivos nuevos.")
    if vectorstore is None:
        print("⚠ No hay base de datos y no se encontraron PDFs para procesar.")
        vectorstore = None

PROCESAMIENTO DE PDFs
📂 Intentando cargar base de datos vectorial existente...
✓ Base de datos vectorial cargada (1024 documentos)
   - PDFs ya procesados: 11
   - Archivos: SerialMurder-PathwaysForInvestigations.pdf, Psychology_of_Child_Serial_Killer.pdf, El-gran-libro-de-los-asesinos-en-serie-primeras-paginas.pdf, EmpiricalTestofHolmesSerialMurderTypology.pdf, Dialnet-DahmerBundyRaderAsesinosEnSerie-5527474.pdf...

📊 Resumen:
   - PDFs en directorio: 11
   - PDFs ya procesados: 11
   - PDFs nuevos detectados: 0

✓ Todos los PDFs ya están procesados. No hay archivos nuevos.


## 5. Sistema LangGraph

Definición del estado del agente y los nodos del grafo para orquestar el flujo RAG.


In [44]:
# Definición del estado del grafo
class GraphState(TypedDict):
    """
    Estado del grafo LangGraph que mantiene:
    - messages: Historial de mensajes de la conversación
    - context: Documentos recuperados relevantes
    - question: Última pregunta del usuario
    """
    messages: Annotated[List[BaseMessage], operator.add]
    context: List[dict]
    question: str

print("✓ Estado del grafo definido")

✓ Estado del grafo definido


In [45]:
# Inicializar LLM de Groq
if GROQ_API_KEY and GROQ_API_KEY != "tu-api-key-de-groq-aqui":
    llm = ChatGroq(
        groq_api_key=GROQ_API_KEY,
        model_name=GROQ_MODEL,
        temperature=GROQ_TEMPERATURE
    )
    print(f"✓ LLM Groq inicializado: {GROQ_MODEL}")
else:
    print("⚠ GROQ_API_KEY no configurada. Por favor, configura tu API key.")
    llm = None

# Template de prompt académico y profesional (se reutiliza en el estado)
PROMPT_TEMPLATE = """Eres un asistente académico especializado en criminología y casos de asesinos seriales.

Tu función es proporcionar información precisa, objetiva y académica basada ÚNICAMENTE en los documentos proporcionados.

CONTEXTO RELEVANTE:
{context}

INSTRUCCIONES:
1. Responde la pregunta del usuario basándote SOLO en la información del contexto proporcionado.
2. Si la información no está disponible en el contexto, indica claramente que no tienes esa información.
3. Mantén un tono académico, objetivo y profesional.
4. Cita las fuentes cuando sea relevante (menciona el documento de origen si es apropiado).
5. Sé preciso con fechas, nombres y detalles.

PREGUNTA DEL USUARIO:
{question}

RESPUESTA:"""

print("✓ Template de prompt definido")

✓ LLM Groq inicializado: llama-3.3-70b-versatile
✓ Template de prompt definido


In [46]:
def retrieve_node(state: GraphState) -> GraphState:
    """
    Nodo de recuperación: Busca documentos relevantes en ChromaDB.
    """
    question = state["question"]
    
    if vectorstore is None:
        state["context"] = []
        return state
    
    # Recuperar documentos relevantes
    relevant_docs = retrieve_relevant_documents(vectorstore, question)
    
    # Formatear contexto
    context_text = "\n\n".join([
        f"[Documento {i+1} - {doc.metadata.get('source_file', 'Unknown')}]\n{doc.page_content}"
        for i, doc in enumerate(relevant_docs)
    ])
    
    state["context"] = [{"content": doc.page_content, "metadata": doc.metadata} 
                        for doc in relevant_docs]
    
    return state


def generate_node(state: GraphState) -> GraphState:
    """
    Nodo de generación: Genera respuesta usando Groq con el contexto recuperado.
    """
    question = state["question"]
    context = state["context"]
    
    # Construir contexto formateado
    context_text = "\n\n".join([
        f"[Fuente: {doc['metadata'].get('source_file', 'Unknown')} - Página {doc['metadata'].get('page', '?')}]\n{doc['content']}"
        for doc in context
    ])
    
    # Crear prompt con el template reutilizable
    prompt = PROMPT_TEMPLATE.format(context=context_text, question=question)
    
    if llm is None:
        response_text = "Error: LLM no configurado. Por favor, configura tu GROQ_API_KEY."
    else:
        try:
            # Generar respuesta
            human_message = HumanMessage(content=prompt)
            response = llm.invoke([human_message])
            response_text = response.content
        except Exception as e:
            response_text = f"Error al generar respuesta: {str(e)}"
    
    # Agregar respuesta al historial de mensajes
    state["messages"].append(AIMessage(content=response_text))
    
    return state

print("✓ Nodos del grafo definidos")

✓ Nodos del grafo definidos


In [47]:
def create_rag_graph() -> StateGraph:
    """
    Crea y compila el grafo LangGraph para el sistema RAG.
    """
    # Crear grafo
    workflow = StateGraph(GraphState)
    
    # Agregar nodos
    workflow.add_node("retrieve", retrieve_node)
    workflow.add_node("generate", generate_node)
    
    # Definir flujo: retrieve -> generate -> END
    workflow.set_entry_point("retrieve")
    workflow.add_edge("retrieve", "generate")
    workflow.add_edge("generate", END)
    
    # Compilar grafo
    app = workflow.compile()
    
    return app

# Crear el grafo (se creará cuando se ejecute)
print("✓ Función para crear grafo definida")

# Nota: El grafo se creará después de que vectorstore esté inicializado

✓ Función para crear grafo definida


## 6. Interfaz Gráfica

Interfaz gráfica moderna y académica para interactuar con el sistema RAG.


In [49]:
# Estilo académico moderno para los widgets
def create_ui():
    """
    Crea la interfaz gráfica del sistema RAG con diseño moderno y profesional.
    """
    # Obtener información del sistema para el header
    doc_count = 0
    if vectorstore is not None:
        try:
            doc_count = vectorstore._collection.count()
        except:
            pass
    
    # Header mejorado con gradiente moderno y mejor diseño
    header_style = widgets.HTML(
        value=f"""
        <div style="
            background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #f093fb 100%);
            color: white;
            padding: 28px 24px;
            border-radius: 16px;
            margin-bottom: 24px;
            box-shadow: 0 8px 24px rgba(102, 126, 234, 0.25), 0 4px 8px rgba(0,0,0,0.1);
            border: 1px solid rgba(255,255,255,0.1);
            position: relative;
            overflow: hidden;
        ">
            <div style="position: relative; z-index: 1;">
                <h1 style="
                    margin: 0 0 8px 0; 
                    font-size: 28px; 
                    font-weight: 700;
                    letter-spacing: -0.5px;
                    text-shadow: 0 2px 4px rgba(0,0,0,0.2);
                    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif;
                ">
                    🎓 Sistema RAG: Agente de Asesinos Seriales
                </h1>
                <p style="
                    margin: 0 0 12px 0; 
                    opacity: 0.95; 
                    font-size: 15px;
                    font-weight: 400;
                    line-height: 1.5;
                ">
                    Sistema de consulta académica basado en documentos PDF
                </p>
                <div style="
                    display: flex;
                    gap: 16px;
                    margin-top: 12px;
                    flex-wrap: wrap;
                ">
                    <span style="
                        background: rgba(255,255,255,0.2);
                        backdrop-filter: blur(10px);
                        padding: 6px 12px;
                        border-radius: 20px;
                        font-size: 12px;
                        font-weight: 500;
                        border: 1px solid rgba(255,255,255,0.3);
                    ">
                        📚 {doc_count} documentos indexados
                    </span>
                    <span style="
                        background: rgba(255,255,255,0.2);
                        backdrop-filter: blur(10px);
                        padding: 6px 12px;
                        border-radius: 20px;
                        font-size: 12px;
                        font-weight: 500;
                        border: 1px solid rgba(255,255,255,0.3);
                    ">
                        🤖 {GROQ_MODEL}
                    </span>
                </div>
            </div>
        </div>
        """,
        layout=widgets.Layout(width='100%', margin='0 0 24px 0')
    )
    
    # Área de consulta mejorada
    query_label = widgets.HTML(
        value="""
        <div style="
            font-size: 16px;
            font-weight: 600;
            color: #2d3748;
            margin-bottom: 8px;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        ">
            💬 Ingrese su consulta:
        </div>
        """,
        layout=widgets.Layout(margin='0 0 8px 0')
    )
    
    query_text = widgets.Textarea(
        value='',
        placeholder='Ejemplo: "¿Quién fue Ted Bundy y cuáles fueron sus crímenes más notorios?"',
        description='',
        disabled=False,
        rows=4,
        layout=widgets.Layout(
            width='100%', 
            margin='0 0 12px 0',
            border='2px solid #e2e8f0',
            padding='12px'
        ),
        style={
            'description_width': '0px',
            'font_size': '14px',
            'font_family': '-apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif'
        }
    )
    
    # Botón de envío mejorado
    submit_button = widgets.Button(
        description='🔍 Consultar',
        button_style='primary',
        layout=widgets.Layout(
            width='180px', 
            height='44px', 
            margin='0 0 24px 0',
            border='none'
        ),
        style={
            'button_color': '#667eea',
            'font_weight': '600',
            'font_size': '15px',
            'box_shadow': '0 4px 12px rgba(102, 126, 234, 0.3)'
        }
    )
    
    # Área de estado mejorada
    status_output = widgets.Output(
        layout=widgets.Layout(
            width='100%', 
            margin='0 0 20px 0',
            padding='12px',
            border='1px solid #e2e8f0',
            border_radius='8px',
            background='#f7fafc'
        )
    )
    
    # Área de respuesta mejorada
    response_label = widgets.HTML(
        value="""
        <div style="
            font-size: 16px;
            font-weight: 600;
            color: #2d3748;
            margin-bottom: 10px;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        ">
            📝 Respuesta:
        </div>
        """,
        layout=widgets.Layout(margin='0 0 8px 0')
    )
    
    response_output = widgets.Output(
        layout=widgets.Layout(
            width='100%',
            min_height='250px',
            border='2px solid #e2e8f0',
            border_radius='12px',
            padding='20px',
            background='#ffffff',
            box_shadow='0 2px 8px rgba(0,0,0,0.04)'
        )
    )
    
    # Historial de conversación mejorado
    history_label = widgets.HTML(
        value="""
        <div style="
            font-size: 16px;
            font-weight: 600;
            color: #2d3748;
            margin: 24px 0 10px 0;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
        ">
            📜 Historial de Conversación:
        </div>
        """,
        layout=widgets.Layout(margin='24px 0 10px 0')
    )
    
    history_output = widgets.Output(
        layout=widgets.Layout(
            width='100%',
            max_height='350px',
            overflow='auto',
            border='2px solid #e2e8f0',
            border_radius='12px',
            padding='16px',
            background='#fafbfc',
            box_shadow='0 2px 8px rgba(0,0,0,0.04)'
        )
    )
    
    # Botón para limpiar historial mejorado
    clear_button = widgets.Button(
        description='🗑️ Limpiar Historial',
        button_style='',
        layout=widgets.Layout(
            width='180px', 
            height='40px', 
            margin='12px 0 0 0',
            border='none'
        ),
        style={
            'button_color': '#e53e3e',
            'font_weight': '600',
            'font_size': '14px',
            'box_shadow': '0 2px 8px rgba(229, 62, 62, 0.2)'
        }
    )
    
    return {
        'header': header_style,
        'query_label': query_label,
        'query_text': query_text,
        'submit_button': submit_button,
        'status_output': status_output,
        'response_label': response_label,
        'response_output': response_output,
        'history_label': history_label,
        'history_output': history_output,
        'clear_button': clear_button
    }

print("✓ Función de interfaz gráfica definida")

✓ Función de interfaz gráfica definida


In [50]:
# Variable global para mantener el estado de la conversación
conversation_history = []
rag_graph = None

def initialize_graph():
    """Inicializa el grafo RAG si aún no está creado."""
    global rag_graph
    if rag_graph is None and vectorstore is not None:
        rag_graph = create_rag_graph()
        print("✓ Grafo RAG inicializado")
    return rag_graph is not None

print("✓ Funciones de inicialización definidas")

✓ Funciones de inicialización definidas


In [51]:
def format_markdown_response(text: str) -> str:
    """Formatea la respuesta para mejor visualización en markdown."""
    # Agregar formato básico
    formatted = text.replace('\n\n', '\n\n')
    return formatted


def process_query(question: str, ui_components: dict):
    """
    Procesa una consulta del usuario usando el sistema RAG con LangGraph.
    """
    global conversation_history, rag_graph
    
    # LIMPIAR HISTORIAL ANTES DE CADA NUEVA BÚSQUEDA
    conversation_history = []
    
    # Limpiar widgets de visualización
    with ui_components['response_output']:
        clear_output(wait=False)
    with ui_components['history_output']:
        clear_output(wait=False)
    with ui_components['status_output']:
        clear_output(wait=False)
    
    # Verificar que el sistema esté inicializado
    if vectorstore is None:
        with ui_components['response_output']:
            display(Markdown("**⚠ Error:** No hay base de datos vectorial disponible. Por favor, procesa los PDFs primero."))
        return
    
    if not initialize_graph():
        with ui_components['response_output']:
            display(Markdown("**⚠ Error:** No se pudo inicializar el grafo RAG."))
        return
    
    if llm is None:
        with ui_components['response_output']:
            display(Markdown("**⚠ Error:** LLM no configurado. Por favor, configura tu GROQ_API_KEY."))
        return
    
    # Actualizar estado inicial
    with ui_components['status_output']:
        print("🔍 Buscando información relevante...")
    
    try:
        # Crear estado inicial
        initial_state = {
            "messages": [HumanMessage(content=question)],
            "context": [],
            "question": question
        }
        
        # Ejecutar el grafo
        final_state = rag_graph.invoke(initial_state)
        
        # Obtener respuesta
        if final_state["messages"]:
            ai_response = final_state["messages"][-1].content
            num_docs = len(final_state.get("context", []))
            
            # Agregar solo esta consulta al historial (ya limpiamos antes)
            conversation_history.append({
                "question": question,
                "answer": ai_response,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            })
            
            # Mostrar respuesta
            with ui_components['response_output']:
                display(Markdown(format_markdown_response(ai_response)))
            
            # Actualizar historial
            update_history_display(ui_components['history_output'])
            
            # Actualizar estado final
            with ui_components['status_output']:
                clear_output(wait=False)
                print(f"✓ Consulta completada exitosamente ({num_docs} documento(s) recuperado(s))")
        else:
            raise Exception("No se generó respuesta")
            
    except Exception as e:
        error_msg = f"Error al procesar consulta: {str(e)}"
        with ui_components['response_output']:
            display(Markdown(f"**❌ Error:** {error_msg}"))
        with ui_components['status_output']:
            clear_output(wait=False)
            print(f"❌ {error_msg}")


def update_history_display(history_output):
    """Actualiza la visualización del historial de conversación."""
    with history_output:
        clear_output(wait=False)
        if conversation_history:
            history_html = "<div style='font-family: Arial, sans-serif;'>"
            for idx, conv in enumerate(conversation_history, 1):
                history_html += f"""
                <div style='margin-bottom: 15px; padding: 10px; background: #f8f9fa; border-left: 3px solid #667eea; border-radius: 3px;'>
                    <div style='font-size: 11px; color: #666; margin-bottom: 5px;'>{conv['timestamp']}</div>
                    <div style='margin-bottom: 8px;'><strong>Q:</strong> {conv['question']}</div>
                    <div style='color: #555;'><strong>A:</strong> {conv['answer'][:200]}...</div>
                </div>
                """
            history_html += "</div>"
            display(Markdown(history_html))


def clear_history(ui_components: dict):
    """Limpia el historial de conversación."""
    global conversation_history
    conversation_history = []
    # Limpiar todos los widgets
    with ui_components['history_output']:
        clear_output(wait=False)
    with ui_components['response_output']:
        clear_output(wait=False)
    with ui_components['status_output']:
        clear_output(wait=False)

print("✓ Funciones de procesamiento de consultas definidas")

✓ Funciones de procesamiento de consultas definidas


In [52]:
# Crear y mostrar la interfaz gráfica
ui = create_ui()

# Configurar eventos
def on_submit_click(b):
    question = ui['query_text'].value.strip()
    if question:
        process_query(question, ui)
        ui['query_text'].value = ''  # Limpiar campo de texto
    else:
        with ui['response_output']:
            clear_output()
            display(Markdown("**⚠ Por favor, ingrese una consulta.**"))

def on_clear_click(b):
    clear_history(ui)

ui['submit_button'].on_click(on_submit_click)
ui['clear_button'].on_click(on_clear_click)

# Mostrar interfaz
display(ui['header'])
display(ui['query_label'])
display(ui['query_text'])
display(ui['submit_button'])
display(ui['status_output'])
display(ui['response_label'])
display(ui['response_output'])
display(ui['history_label'])
display(ui['history_output'])
display(ui['clear_button'])

print("\n" + "="*60)
print("🎉 INTERFAZ GRÁFICA LISTA")
print("="*60)
print("\nIngrese su consulta en el campo de texto y haga clic en 'Consultar'.")
print("Asegúrese de haber:")
print("  1. Configurado su GROQ_API_KEY")
print("  2. Procesado los PDFs (celda anterior)")
print("  3. Inicializado el sistema correctamente")
print("\n" + "="*60)

HTML(value='\n        <div style="\n            background: linear-gradient(135deg, #667eea 0%, #764ba2 50%, #…

HTML(value='\n        <div style="\n            font-size: 16px;\n            font-weight: 600;\n            c…

Textarea(value='', layout=Layout(border_bottom='2px solid #e2e8f0', border_left='2px solid #e2e8f0', border_ri…

Button(button_style='primary', description='🔍 Consultar', layout=Layout(border_bottom='none', border_left='non…

Output(layout=Layout(border_bottom='1px solid #e2e8f0', border_left='1px solid #e2e8f0', border_right='1px sol…

...

Output(layout=Layout(border_bottom='2px solid #e2e8f0', border_left='2px solid #e2e8f0', border_right='2px sol…

HTML(value='\n        <div style="\n            font-size: 16px;\n            font-weight: 600;\n            c…

Output(layout=Layout(border_bottom='2px solid #e2e8f0', border_left='2px solid #e2e8f0', border_right='2px sol…

Button(description='🗑️ Limpiar Historial', layout=Layout(border_bottom='none', border_left='none', border_righ…


🎉 INTERFAZ GRÁFICA LISTA

Ingrese su consulta en el campo de texto y haga clic en 'Consultar'.
Asegúrese de haber:
  1. Configurado su GROQ_API_KEY
  2. Procesado los PDFs (celda anterior)
  3. Inicializado el sistema correctamente

